In [1]:
!pip install networkx

In [2]:
import pandas as pd
import networkx as nx

CSV_FILE = "HI-Small_Trans.csv"
OUTPUT_FILE = "AML_sample.gexf"

# How many laundering transactions to use
N_LAUNDERING = 100

# Number of surrounding normal transactions to include
N_NORMAL = 300

# --------------------------------------------------
# STEP 1: Find laundering transactions
# --------------------------------------------------

laundering_rows = []

for chunk in pd.read_csv(CSV_FILE, chunksize=100_000):

    laundering = chunk[
        chunk["Is Laundering"] == 1
    ]

    if len(laundering) > 0:
        laundering_rows.append(laundering)

    # Stop once we have enough
    current_count = sum(len(x) for x in laundering_rows)

    if current_count >= N_LAUNDERING:
        break


laundering_df = pd.concat(laundering_rows)

laundering_df = laundering_df.head(N_LAUNDERING)

print("Laundering transactions:", len(laundering_df))


# --------------------------------------------------
# STEP 2: Get the accounts involved
# --------------------------------------------------

accounts = set(
    laundering_df["Account"]
).union(
    set(laundering_df["Account.1"])
)

print("Accounts involved:", len(accounts))


# --------------------------------------------------
# STEP 3: Find transactions between these accounts
# --------------------------------------------------

normal_rows = []

for chunk in pd.read_csv(CSV_FILE, chunksize=100_000):

    # Transactions where BOTH accounts belong
    # to the laundering network
    mask = (
        chunk["Account"].isin(accounts)
        &
        chunk["Account.1"].isin(accounts)
    )

    normal = chunk[mask]

    # Don't include laundering rows here
    normal = normal[
        normal["Is Laundering"] == 0
    ]

    if len(normal) > 0:
        normal_rows.append(normal)

    current_count = sum(len(x) for x in normal_rows)

    if current_count >= N_NORMAL:
        break


if normal_rows:
    normal_df = pd.concat(normal_rows).head(N_NORMAL)
else:
    normal_df = pd.DataFrame()


print("Normal transactions:", len(normal_df))


# --------------------------------------------------
# STEP 4: Create graph
# --------------------------------------------------

G = nx.MultiDiGraph()


# Add laundering transactions
for _, row in laundering_df.iterrows():

    sender = str(row["Account"])
    receiver = str(row["Account.1"])

    G.add_node(sender)
    G.add_node(receiver)

    G.add_edge(
        sender,
        receiver,
        amount=float(row["Amount Received"]),
        laundering=1,
        transaction_type="laundering"
    )


# Add normal transactions
for _, row in normal_df.iterrows():

    sender = str(row["Account"])
    receiver = str(row["Account.1"])

    G.add_node(sender)
    G.add_node(receiver)

    G.add_edge(
        sender,
        receiver,
        amount=float(row["Amount Received"]),
        laundering=0,
        transaction_type="normal"
    )


# --------------------------------------------------
# STEP 5: Save for Gephi
# --------------------------------------------------

nx.write_gexf(G, OUTPUT_FILE)

print("Graph created!")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("File:", OUTPUT_FILE)

Laundering transactions: 100
Accounts involved: 128
Normal transactions: 300
Graph created!
Nodes: 128
Edges: 400
File: AML_sample.gexf
